<a href="https://colab.research.google.com/github/MadushaniWijesooriya/My-Project-01-Employee-managemet-System-/blob/main/ML_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Load original data
df = pd.read_csv('/content/courses_dataset.csv')
print("Original Shape:", df.shape)

#  DATA AUGMENTATION
def simple_augment(text):
    text = str(text).strip()
    variations = [
        text,
        text.replace("?", ""),
        text + " please",
        "Can you tell me " + text.lower(),
        text.replace("course", "program"),
        text.replace("course", "class")
    ]
    return variations

# Augment all minority classes
augmented_rows = []
minority_intents = ['apply', 'contact', 'ai_course', 'english_course']

for intent in minority_intents:
    subset = df[df['intent'] == intent].copy()
    for _, row in subset.iterrows():
        aug_texts = simple_augment(row['question'])
        for aug_text in aug_texts:
            if aug_text not in df['question'].values:
                augmented_rows.append({'question': aug_text, 'intent': intent})

aug_df = pd.DataFrame(augmented_rows)
df = pd.concat([df, aug_df], ignore_index=True).drop_duplicates(subset=['question'])

print("After Augmentation Shape:", df.shape)
print(df['intent'].value_counts())

#  PREPROCESSING
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_question'] = df['question'].apply(clean_text)

#  VECTORIZER & MODEL
vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    max_df=0.85,
    min_df=1,
    max_features=10000
)

X = vectorizer.fit_transform(df['clean_question'])
y = df['intent']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(
    class_weight='balanced',
    C=30,
    max_iter=3000,
    solver='liblinear',
    random_state=42
)

model.fit(X_train, y_train)

#  RESULTS
y_pred = model.predict(X_test)
print("\n=== FINAL ACCURACY ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\n", classification_report(y_test, y_pred, zero_division=0))

# Save Model
joblib.dump(model, 'model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
print("\n✅ Improved Model Saved Successfully!")

In [ ]:
from flask import Flask, render_template, request, jsonify
import joblib
import os

app = Flask(__name__)

# Load Model
try:
    model = joblib.load('model.pkl')
    vectorizer = joblib.load('vectorizer.pkl')
    print("✅ Model loaded successfully!")
except:
    model = None
    vectorizer = None
    print("⚠️ Model not found. Please run training first.")

# Responses
responses = {
    'ai_course': "The AI Applications in Office Operations course is 5 weeks (30 hours). Lectures on Sundays. Fee: LKR 8,500.",
    'english_course': "The English & Computer Basics for School Leavers course is 16 weeks (126 hours). Lectures on Saturdays. Fee: LKR 25,000.",
    'general': "ATI Gampaha offers two excellent short-term courses to enhance your skills.",
    'apply': "You can apply online using this link: https://forms.gle/8Rknmy7dBNSvqKLo7",
    'contact': "Contact ATI Gampaha: 033-2292544 | atigampaha@sliate.ac.lk | Naiwala, Essella, Veyangoda."
}

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/chat', methods=['POST'])
def chat():
    try:
        user_message = request.json.get('message', '')

        if model and vectorizer:
            vec = vectorizer.transform([user_message])
            intent = model.predict(vec)[0]
            reply = responses.get(intent, responses['general'])
        else:
            reply = "The chatbot is not trained yet. Please train the model first."

        return jsonify({'reply': reply})
    except:
        return jsonify({'reply': "Sorry, I didn't understand that."})

if __name__ == '__main__':
    if not os.path.exists('templates'):
        os.makedirs('templates')
    print("🚀 Chatbot is running...")
    app.run(host='0.0.0.0', port=5000)

✅ Model loaded successfully!
🚀 Chatbot is running...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


In [ ]:
!pip install flask-ngrok